# IR System — Hybrid Retrieval
**Step 7:** Two hybrid strategies:
- **Parallel (RRF):** fuse BM25 + TF-IDF + Embedding results
- **Serial:** BM25 first-stage → Embedding re-rank

⚠️ Use **GPU runtime** (Runtime → Change runtime type → T4 GPU)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/ir_system_data'
import os, sys

if not os.path.exists('/content/ir-system'):
    !git clone https://github.com/ghazal-mohammad/ir-system.git /content/ir-system
else:
    !cd /content/ir-system && git pull
sys.path.insert(0, '/content/ir-system')

!pip install ir-datasets==0.5.9 sentence-transformers==2.7.0 -q
print('ready')

In [ ]:
import json
import ir_datasets
from services.preprocessing_service import preprocess
from services.indexing_service import load_index, get_avg_doc_length
from services.bm25_service import retrieve_bm25, load_bm25_params
from services.embedding_service import load_model, load_embeddings, retrieve_embedding
from services.hybrid_service import hybrid_parallel, hybrid_serial

# Load CT2021 data
print('Loading CT2021 data...')
index1 = load_index(f'{SAVE_DIR}/ct2021_index.pkl')
with open(f'{SAVE_DIR}/ct2021_doc_lengths.json') as f:
    doc_lengths1 = json.load(f)
bm25_params1 = load_bm25_params(f'{SAVE_DIR}/ct2021_bm25_params.pkl')
avg_dl1 = bm25_params1['avg_dl']
doc_ids1, emb1 = load_embeddings(f'{SAVE_DIR}/ct2021')
print(f'CT2021 loaded — {len(doc_ids1):,} docs, embeddings {emb1.shape}')

In [ ]:
# Load model + queries
print('Loading model...')
model = load_model()

ds1 = ir_datasets.load('clinicaltrials/2021/trec-ct-2021')
queries1 = {q.query_id: q.text for q in ds1.queries_iter()}
print(f'CT2021: {len(queries1)} queries')

In [ ]:
# Test all hybrid methods on one query
sample_qid = list(queries1.keys())[0]
sample_query = queries1[sample_qid]
tokens = preprocess(sample_query)

bm25_res = retrieve_bm25(tokens, index1, doc_lengths1, avg_dl1, top_k=1000)
emb_res  = retrieve_embedding(sample_query, model, doc_ids1, emb1, top_k=1000)

# Load TF-IDF results from pre-computed JSON
with open(f'{SAVE_DIR}/ct2021_tfidf_results.json') as f:
    tfidf_all1 = json.load(f)
tfidf_res = tfidf_all1.get(sample_qid, [])

rrf_res    = hybrid_parallel(bm25_res, tfidf_res, emb_res, top_k=10)
serial_res = hybrid_serial(sample_query, bm25_res, model, doc_ids1, emb1,
                            first_stage_k=100, top_k=10)

print(f'Query: "{sample_query}"')
print('\n--- RRF Parallel Top 10 ---')
for r in rrf_res:
    print(f'  rank {r["rank"]}: {r["doc_id"]} (score={r["score"]})')
print('\n--- Serial (BM25→Embedding rerank) Top 10 ---')
for r in serial_res:
    print(f'  rank {r["rank"]}: {r["doc_id"]} (score={r["score"]})')

In [ ]:
# Full hybrid retrieval for CT2021
print('Loading pre-computed results for CT2021...')
with open(f'{SAVE_DIR}/ct2021_bm25_results.json') as f:
    bm25_all1 = json.load(f)
with open(f'{SAVE_DIR}/ct2021_embedding_results.json') as f:
    emb_all1 = json.load(f)

print('Running Hybrid on CT2021...')
hybrid_parallel_all1 = {}
hybrid_serial_all1   = {}

for qid, qtext in queries1.items():
    b = bm25_all1.get(qid, [])
    t = tfidf_all1.get(qid, [])
    e = emb_all1.get(qid, [])
    hybrid_parallel_all1[qid] = hybrid_parallel(b, t, e, top_k=1000)
    hybrid_serial_all1[qid]   = hybrid_serial(qtext, b, model, doc_ids1, emb1,
                                               first_stage_k=100, top_k=1000)

with open(f'{SAVE_DIR}/ct2021_hybrid_parallel_results.json', 'w') as f:
    json.dump(hybrid_parallel_all1, f)
with open(f'{SAVE_DIR}/ct2021_hybrid_serial_results.json', 'w') as f:
    json.dump(hybrid_serial_all1, f)
print(f'CT2021 hybrid done: {len(hybrid_parallel_all1)} queries')

In [ ]:
# Full hybrid retrieval for MSMARCO
print('Loading MSMARCO data...')
index2 = load_index(f'{SAVE_DIR}/msmarco_index.pkl')
with open(f'{SAVE_DIR}/msmarco_doc_lengths.json') as f:
    doc_lengths2 = json.load(f)
bm25_params2 = load_bm25_params(f'{SAVE_DIR}/msmarco_bm25_params.pkl')
avg_dl2 = bm25_params2['avg_dl']
doc_ids2, emb2 = load_embeddings(f'{SAVE_DIR}/msmarco')

ds2 = ir_datasets.load('msmarco-passage/trec-dl-2019')
queries2 = {q.query_id: q.text for q in ds2.queries_iter()}

with open(f'{SAVE_DIR}/msmarco_bm25_results.json') as f:
    bm25_all2 = json.load(f)
with open(f'{SAVE_DIR}/msmarco_tfidf_results.json') as f:
    tfidf_all2 = json.load(f)
with open(f'{SAVE_DIR}/msmarco_embedding_results.json') as f:
    emb_all2 = json.load(f)

print('Running Hybrid on MSMARCO...')
hybrid_parallel_all2 = {}
hybrid_serial_all2   = {}

for qid, qtext in queries2.items():
    b = bm25_all2.get(qid, [])
    t = tfidf_all2.get(qid, [])
    e = emb_all2.get(qid, [])
    hybrid_parallel_all2[qid] = hybrid_parallel(b, t, e, top_k=1000)
    hybrid_serial_all2[qid]   = hybrid_serial(qtext, b, model, doc_ids2, emb2,
                                               first_stage_k=100, top_k=1000)

with open(f'{SAVE_DIR}/msmarco_hybrid_parallel_results.json', 'w') as f:
    json.dump(hybrid_parallel_all2, f)
with open(f'{SAVE_DIR}/msmarco_hybrid_serial_results.json', 'w') as f:
    json.dump(hybrid_serial_all2, f)
print(f'MSMARCO hybrid done: {len(hybrid_parallel_all2)} queries')

print('\n=== Hybrid Retrieval Complete ===')
print('Next: 08_evaluation.ipynb')